# Customer Churn Analysis

## Business Problem

Customer churn is a major problem for subscription businesses. Losing customers increases acquisition costs and reduces revenue.

This project analyzes customer data behavior to identify factors associated with churn and build a predictive model to identify customers at risk of leaving.

## Objective

1. Understand customer churn patterns through exploratory data analysis.
2. Identify key factors that influence customer churn.
3. Build an interpretable churn prediction model.
4. Translate model findings into actionable retention strategies.

## Dataset

**Dataset:**

https://www.kaggle.com/datasets/blastchar/telco-customer-churn

**Target Variable:**

`Churn`

**Goal:**

Predict whether a customer is likely to leave the service.

# 1. Import Libraries

We import the main Python Libraries needed for:

- Data manipulation
- Data visualization
- Machine learning preparation


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

print("Libraries imported successfully")

Libraries imported successfully


# 2. Load Dataset

The first step is to load the customer churn dataset and understand its structure.

We will check:

- Number of rows and colunms
- Colunm names
- Data types
- Missing values
- Basic statistics

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from churn_analysis import clean_customer_data, evaluate_binary_classifier

data_path = "../data/raw/Telco-Customer-Churn.csv"
df = pd.read_csv(data_path)

df.drop(columns="customerID").head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.shape

(7043, 21)

# 3. Dataset Overview

Before analysis, we need to understand:

- Colunm data types
- Missing values
- Potential data quality issues

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [5]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [6]:
(df == " ").sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

# 4. Data Cleaning

The raw data contains 11 blank values in `TotalCharges`. These are converted to missing values and removed before analysis, leaving 7,032 records.

In [7]:
df = clean_customer_data(df)

df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [8]:
df.shape

(7032, 21)

In [9]:
df.duplicated().sum()

np.int64(0)

# 5. Target Variable Analysis

Understanding the distribution of churn is important because it determines:

- How balanced the dataset is
- Which evaluation metrics we should prioritize
- How common customer churn is

In [10]:
df["Churn"].value_counts()

Churn
No     5163
Yes    1869
Name: count, dtype: int64

In [11]:
plt.figure(figsize=(6,4))

sns.countplot(
    data=df,
    x="Churn"
)

plt.title("Customer Churn Distribution")
plt.xlabel("Churn Status")
plt.ylabel("Number of Customers")

plt.show()

In [12]:
df["Churn"].value_counts(normalize=True)

Churn
No     0.734215
Yes    0.265785
Name: proportion, dtype: float64

# 6. Exploratory Data Analysis (EDA)

Exploratory Data Analysis helps us understand customer behavior and identify
patterns associated with churn.

We will analyze churn across:

- Customer demographics
- Service usage
- Contract information
- Payment behavior
- Charges

## How does churn rate vary by contract type?

In [13]:
contract_churn = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
)

contract_churn

Churn,No,Yes
Contract,,
Month-to-month,0.572903,0.427097
One year,0.887228,0.112772
Two year,0.971513,0.028487


## Contract Type and Observed Churn

Month-to-month customers have the highest observed churn rate (42.7%); one-year contracts have 11.3%, and two-year contracts have 2.8%. These are descriptive associations and do not show that changing contract terms would cause churn to fall. Contract offers would need to be tested.

In [14]:
contract_churn["Yes"].plot(
    kind="bar",
    figsize=(7,4)
)

plt.title("Churn Rate by Contract Type")
plt.ylabel("Churn Rate")
plt.xlabel("Contract Type")
plt.xticks(rotation=0)

plt.show()

## How does observed churn vary by tenure?

In [15]:
df["tenure"].describe()

count    7032.000000
mean       32.421786
std        24.545260
min         1.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: tenure, dtype: float64

## Customer Tenure Segmentation

To better understand churn behavior, we group customers into tenure ranges.

This helps identify whether new customers or long-term customers are more likely to churn.

In [16]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=[
        "0-12 months",
        "13-24 months",
        "25-48 months",
        "49-72 months"
    ]
)

df["tenure_group"].value_counts()

tenure_group
49-72 months    2239
0-12 months     2175
25-48 months    1594
13-24 months    1024
Name: count, dtype: int64

In [17]:
tenure_churn = pd.crosstab(
    df["tenure_group"],
    df["Churn"],
    normalize="index"
)

tenure_churn

Churn,No,Yes
tenure_group,,
0-12 months,0.523218,0.476782
13-24 months,0.712891,0.287109
25-48 months,0.796110,0.203890
49-72 months,0.904868,0.095132


## Tenure and Observed Churn

Customers in their first 12 months have a 47.7% observed churn rate, compared with 9.5% for customers with 49–72 months of tenure. This is a segment difference in the snapshot, not evidence that tenure itself causes retention.

In [18]:
tenure_churn["Yes"].plot(
    kind="bar",
    figsize=(7,4)
)

plt.title("Churn Rate by Customer Tenure")
plt.ylabel("Churn Rate")
plt.xlabel("Tenure Group")
plt.xticks(rotation=45)

plt.show()

## How does observed churn vary by payment method?

In [19]:
payment_churn = pd.crosstab(
    df["PaymentMethod"],
    df["Churn"],
    normalize="index"
)

payment_churn

Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),0.832685,0.167315
Credit card (automatic),0.847469,0.152531
Electronic check,0.547146,0.452854
Mailed check,0.807980,0.192020


In [20]:
payment_churn["Yes"].plot(
    kind="bar",
    figsize=(8,4)
)

plt.title("Churn Rate by Payment Method")
plt.ylabel("Churn Rate")
plt.xlabel("Payment Method")
plt.xticks(rotation=45, ha="right")

plt.show()

## Payment Method and Observed Churn

Electronic-check customers have a 45.3% observed churn rate, compared with 16.7% for automatic bank transfer and 15.3% for automatic credit card. The data does not establish why these differences occur; billing friction is one hypothesis to investigate.

## How do monthly charges differ by churn outcome?

In [21]:
plt.figure(figsize=(8,4))

sns.boxplot(
    data=df,
    x="Churn",
    y="MonthlyCharges"
)

plt.title("Monthly Charges by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")

plt.show()

In [22]:
df.groupby("Churn")["MonthlyCharges"].mean()

Churn
No     61.307408
Yes    74.441332
Name: MonthlyCharges, dtype: float64

## Monthly Charges and Observed Churn

Customers who churn have higher average monthly charges (74.44) than customers who stay (61.31). No price-sensitivity or perceived-value measure is present, so those mechanisms cannot be inferred from this comparison.

## How does observed churn vary by technical support?

In [23]:
tech_support_churn = pd.crosstab(
    df["TechSupport"],
    df["Churn"],
    normalize="index"
)

tech_support_churn

Churn,No,Yes
TechSupport,,
No,0.583525,0.416475
No internet service,0.925658,0.074342
Yes,0.848039,0.151961


In [24]:
tech_support_churn["Yes"].plot(
    kind="bar",
    figsize=(7,4)
)

plt.title("Churn Rate by Technical Support")
plt.ylabel("Churn Rate")
plt.xlabel("Technical Support")
plt.xticks(rotation=0)

plt.show()

## Technical Support and Observed Churn

Customers without technical support have a 41.6% observed churn rate, compared with 15.2% among customers with support. This association may reflect other customer or service differences and does not establish a causal effect of support.

# EDA Summary: Key Churn Drivers

The snapshot shows differences in churn rates across several customer segments; these are associations, not causal effects.

## Main Findings

1. **Contract Type**

   Month-to-month customers have the highest churn rate (42.7%), while customers with two-year contracts have a much lower churn rate (2.8%). Longer customer commitments are associated with better retention.

2. **Customer Tenure**

   Customers in their first 12 months have the highest churn risk, with a churn rate of 47.7%. Early customer experience appears to be a critical retention period.

3. **Payment Method** 

   Customers using electronic checks have a higher churn rate (45.3%) compared with customers using automatic payment methods. Payment behavior may indicate differences in customer engagement and billing experience.

4. **Monthly Charges**

   Customers who churn have higher average monthly charges ($74.44) compared with customers who stay ($61.31). Higher-paying customers may require additional attention to ensure they receive enough value.

5. **Technical Support**

   Customers without technical support have a higher churn rate (41.6%) compared with customers who have technical support (15.2%). Support availability is associated with stronger customer retention.

## Modeling Direction

These findings are descriptive and motivate hypotheses for future validation:

- Use customer demographics, services, contract details, and billing information as model features.
- Build a classification model to predict customers with high churn risk.
- Analyze feature importance to identify the strongest churn predictors.

# 7. Data Preparation for Modeling

Before building machine learning mdoels, we prepare the dataset my removing irrelevent colunms, encoding categorical variables,
and splitting the data into training and testing sets.

In [25]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'tenure_group'],
      dtype='str')

In [26]:
df = df.drop("customerID", axis=1)

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-12 months
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-48 months
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-12 months
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,25-48 months
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-12 months


In [27]:
df.shape

(7032, 21)

## Convert Target Variable

Machine learning models require numerical labels.

We convert churn status:
- No → 0
- Yes → 1

In [28]:
df["Churn"] = df["Churn"].map(
    {
        "No": 0,
        "Yes": 1
    }
)

df["Churn"].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

## Separate Features and Target

The target variable is customer churn.

All remaining customer attributes will be used as input features for prediction.

In [29]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X.shape, y.shape

((7032, 20), (7032,))

## Split Training and Testing Data

We split the dataset into training and testing sets.

The training data is used to build the model, while the testing data evaluates how well the model performs on unseen customers.

Stratification is used to maintain the same churn distribution in both datasets.

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((5625, 20), (1407, 20))

In [31]:
y_train.value_counts(normalize=True)

Churn
0    0.734222
1    0.265778
Name: proportion, dtype: float64

In [32]:
y_test.value_counts(normalize=True)

Churn
0    0.734186
1    0.265814
Name: proportion, dtype: float64

In [33]:
X.dtypes

gender                   str
SeniorCitizen          int64
Partner                  str
Dependents               str
tenure                 int64
PhoneService             str
MultipleLines            str
InternetService          str
OnlineSecurity           str
OnlineBackup             str
DeviceProtection         str
TechSupport              str
StreamingTV              str
StreamingMovies          str
Contract                 str
PaperlessBilling         str
PaymentMethod            str
MonthlyCharges       float64
TotalCharges         float64
tenure_group        category
dtype: object

In [34]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "tenure_group"
]

In [35]:
len(numeric_features), len(categorical_features)

(4, 16)

## Feature Preprocessing

Machine learning models require numerical input.

Numeric features are kept unchanged, while categorical features are converted using one-hot encoding.

A preprocessing pipeline ensures that the same transformations are applied consistently during training and testing.

In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

# 8. Baseline Model — Logistic Regression

Logistic Regression is used as the first baseline model because it is interpretable and provides insight into which customer characteristics are associated with churn risk.

In [37]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000
            )
        )
    ]
)

## Train Baseline Model

The model learns patterns from the training data.

The test dataset remains unseen and will be used later to evaluate performance.

In [38]:
logistic_model.fit(
    X_train,
    y_train
)
print("Model training completed successfully")

Model training completed successfully


## Model Evaluation

We evaluate the model using multiple metrics because churn data is imbalanced.

Accuracy alone can be misleading, so we also examine precision, recall, F1-score, and ROC-AUC.

In [39]:
y_pred_proba = logistic_model.predict_proba(X_test)[:, 1]

In [40]:
baseline_metrics, y_pred = evaluate_binary_classifier(y_test, y_pred_proba)
accuracy = baseline_metrics["Accuracy"]
precision = baseline_metrics["Precision"]
recall = baseline_metrics["Recall"]
f1 = baseline_metrics["F1 Score"]
roc_auc = baseline_metrics["ROC-AUC"]
average_precision = baseline_metrics["Average Precision"]

accuracy, precision, recall, f1, roc_auc, average_precision

(0.7938877043354655,
 0.635483870967742,
 0.5267379679144385,
 0.5760233918128655,
 0.8349092255048636,
 0.6270814789359299)

## Logistic Regression Results

At the default 0.5 threshold, the model's holdout recall is 52.7% and precision is 63.5%; its ROC-AUC is 83.5%. These describe one stratified test split. The precision-recall analysis below illustrates how a lower threshold changes the number of customers flagged and the precision/recall balance.

In [41]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Stayed", "Churned"]
)

disp.plot()

plt.show()

## Confusion Matrix Analysis

The confusion matrix shows how the model performs when classifying customers as churned or retained.

The model correctly identifies many customers who remain, but some customers who eventually churn are missed.

Reducing false negatives is important because these represent customers who are likely to leave but are not identified early enough for retention actions.

In [42]:
feature_names = logistic_model.named_steps["preprocessor"].get_feature_names_out()
feature_coefficients = logistic_model.named_steps["model"].coef_[0]

feature_importance = pd.DataFrame(
    {"feature": feature_names, "coefficient": feature_coefficients}
)

feature_importance_sorted = feature_importance.sort_values(
    by="coefficient"
)

feature_importance_sorted.head(10)

,feature,coefficient
1,num__tenure,-1.241924
38,cat__Contract_Two year,-0.851278
15,cat__InternetService_DSL,-0.605167
46,cat__tenure_group_13-24 months,-0.451500
2,num__MonthlyCharges,-0.422910
12,cat__MultipleLines_No,-0.309872
39,cat__PaperlessBilling_No,-0.295172
25,cat__DeviceProtection_No internet service,-0.274539
31,cat__StreamingTV_No internet service,-0.274539
28,cat__TechSupport_No internet service,-0.274539


In [43]:
feature_importance_sorted.tail(10)

,feature,coefficient
14,cat__MultipleLines_Yes,0.108168
27,cat__TechSupport_No,0.132460
18,cat__OnlineSecurity_No,0.157614
43,cat__PaymentMethod_Electronic check,0.172040
35,cat__StreamingMovies_Yes,0.180364
32,cat__StreamingTV_Yes,0.199001
3,num__TotalCharges,0.310879
48,cat__tenure_group_49-72 months,0.415263
16,cat__InternetService_Fiber optic,0.582200
36,cat__Contract_Month-to-month,0.677997


## Feature Importance: Factors Associated With Churn

Logistic Regression coefficients help explain which customer characteristics are associated with higher or lower churn probability.

Positive coefficients indicate factors associated with increased churn risk, while negative coefficients indicate factors associated with customer retention.

In [44]:
top_features = pd.concat(
    [
        feature_importance_sorted.head(10),
        feature_importance_sorted.tail(10)
    ]
)

plt.figure(figsize=(10,8))

sns.barplot(
    data=top_features,
    x="coefficient",
    y="feature"
)

plt.xlabel("Logistic Regression Coefficient")
plt.ylabel("Feature")

plt.show()

## Logistic Regression Coefficients

Coefficients describe conditional associations in this fitted Logistic Regression model and depend on encoding and reference categories. They are not causal effects. Month-to-month contract and fiber-optic indicators have positive coefficients; tenure and two-year contract indicators have negative coefficients in this specification.

In [45]:
X_clean = X.drop("tenure_group", axis=1)

In [46]:
X_clean.shape

(7032, 19)

In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [48]:
X_train.shape, X_test.shape

((5625, 19), (1407, 19))

In [49]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [50]:
len(numeric_features), len(categorical_features)

(4, 15)

# 9. Model Improvement

The baseline model included the engineered `tenure_group` feature that was created for exploratory analysis.

Since `tenure` already contains the same information at a finer level of detail, we remove `tenure_group` to reduce feature redundancy and improve model interpretability.

The improved model is trained using the refined feature set and compared against the baseline model.

In [51]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

improved_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [52]:
improved_preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [53]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

improved_logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            improved_preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000
            )
        )
    ]
)

In [54]:
improved_logistic_model.fit(
    X_train,
    y_train
)

print("Improved Logistic Regression model trained successfully.")

Improved Logistic Regression model trained successfully.


In [55]:
improved_y_pred_proba = improved_logistic_model.predict_proba(X_test)[:, 1]

In [56]:
improved_metrics, improved_y_pred = evaluate_binary_classifier(
    y_test,
    improved_y_pred_proba,
)
improved_accuracy = improved_metrics["Accuracy"]
improved_precision = improved_metrics["Precision"]
improved_recall = improved_metrics["Recall"]
improved_f1 = improved_metrics["F1 Score"]
improved_roc_auc = improved_metrics["ROC-AUC"]
improved_average_precision = improved_metrics["Average Precision"]

(
    improved_accuracy,
    improved_precision,
    improved_recall,
    improved_f1,
    improved_roc_auc,
    improved_average_precision,
)

(0.8038379530916845,
 0.6484848484848484,
 0.5721925133689839,
 0.6079545454545454,
 0.8359290473207676,
 0.6228170767081351)

## Model Improvement Results

Removing the engineered `tenure_group` feature simplifies the model because it is derived from `tenure`. On this one holdout split, the refined model's metrics are modestly higher; this single comparison does not establish improved performance on future data.

In [57]:
comparison = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1 Score",
            "ROC-AUC",
            "Average Precision",
        ],
        "Baseline": [
            accuracy,
            precision,
            recall,
            f1,
            roc_auc,
            average_precision,
        ],
        "Improved": [
            improved_accuracy,
            improved_precision,
            improved_recall,
            improved_f1,
            improved_roc_auc,
            improved_average_precision,
        ],
    }
)

comparison

,Metric,Baseline,Improved
0,Accuracy,0.793888,0.803838
1,Precision,0.635484,0.648485
2,Recall,0.526738,0.572193
3,F1 Score,0.576023,0.607955
4,ROC-AUC,0.834909,0.835929
5,Average Precision,0.627081,0.622817


### Precision-Recall and Decision Threshold

The precision-recall curve shows the precision/recall trade-off for the refined Logistic Regression scores. The table compares illustrative thresholds; lowering the threshold flags more customers and can increase recall while reducing precision. The 0.5 threshold is retained for model comparisons because no intervention capacity or error costs were provided, not because it is known to be optimal.

In [58]:
from IPython.display import display
from sklearn.metrics import precision_recall_curve

threshold_rows = []
for threshold in (0.3, 0.5, 0.7):
    threshold_metrics, threshold_predictions = evaluate_binary_classifier(
        y_test,
        improved_y_pred_proba,
        threshold=threshold,
    )
    threshold_rows.append(
        {
            "Threshold": threshold,
            "Precision": threshold_metrics["Precision"],
            "Recall": threshold_metrics["Recall"],
            "F1 Score": threshold_metrics["F1 Score"],
            "Predicted churn share": threshold_predictions.mean(),
        }
    )

threshold_comparison = pd.DataFrame(threshold_rows)
display(threshold_comparison)

pr_precision, pr_recall, _ = precision_recall_curve(y_test, improved_y_pred_proba)
plt.figure(figsize=(7, 5))
plt.plot(
    pr_recall,
    pr_precision,
    label=f"Improved Logistic Regression (AP={improved_average_precision:.3f})",
)
plt.axhline(
    y=y_test.mean(),
    linestyle="--",
    color="gray",
    label="Churn prevalence",
)
plt.scatter(
    [improved_recall],
    [improved_precision],
    marker="o",
    color="black",
    label="Threshold = 0.5",
    zorder=3,
)
plt.title("Precision-Recall Curve: Improved Logistic Regression")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

,Threshold,Precision,Recall,F1 Score,Predicted churn share
0,0.3,0.510791,0.759358,0.610753,0.395167
1,0.5,0.648485,0.572193,0.607955,0.234542
2,0.7,0.759036,0.168449,0.275711,0.058991


## Logistic Regression Feature Comparison

The table compares the two Logistic Regression specifications on the same stratified test split and at the same 0.5 threshold. ROC-AUC and Average Precision use predicted probabilities and do not depend on that classification threshold.

# 10. Model Comparison

A most-frequent DummyClassifier, two Logistic Regression specifications, and a Random Forest are compared using the same training and test rows. Threshold-based metrics use the explicit `probability >= 0.5` rule; ROC-AUC and Average Precision use scores.

In [59]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

In [60]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)
dummy_y_score = dummy_model.predict_proba(X_test)[:, 1]
dummy_metrics, dummy_y_pred = evaluate_binary_classifier(y_test, dummy_y_score)
dummy_metrics

{'Threshold': 0.5,
 'Accuracy': 0.7341862117981521,
 'Precision': 0.0,
 'Recall': 0.0,
 'F1 Score': 0.0,
 'ROC-AUC': 0.5,
 'Average Precision': 0.2658137882018479}

In [61]:
random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            improved_preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42
            )
        )
    ]
)

random_forest_model.fit(
    X_train,
    y_train
)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [62]:
rf_y_pred = random_forest_model.predict(X_test)

rf_y_pred_proba = random_forest_model.predict_proba(X_test)[:, 1]

In [63]:
rf_metrics, rf_y_pred = evaluate_binary_classifier(y_test, rf_y_pred_proba)
rf_accuracy = rf_metrics["Accuracy"]
rf_precision = rf_metrics["Precision"]
rf_recall = rf_metrics["Recall"]
rf_f1 = rf_metrics["F1 Score"]
rf_roc_auc = rf_metrics["ROC-AUC"]
rf_average_precision = rf_metrics["Average Precision"]

(
    rf_accuracy,
    rf_precision,
    rf_recall,
    rf_f1,
    rf_roc_auc,
    rf_average_precision,
)

(0.7846481876332623,
 0.6187290969899666,
 0.4946524064171123,
 0.549777117384844,
 0.8136612120867004,
 0.5942004610222683)

In [64]:
metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC",
    "Average Precision",
]
model_metrics = {
    "Dummy (most frequent)": dummy_metrics,
    "Logistic Regression (with tenure group)": baseline_metrics,
    "Logistic Regression (without tenure group)": improved_metrics,
    "Random Forest": rf_metrics,
}
model_comparison = pd.DataFrame.from_dict(model_metrics, orient="index")[metric_columns]
model_comparison

,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Average Precision
Dummy (most frequent),0.734186,0.000000,0.000000,0.000000,0.500000,0.265814
Logistic Regression (with tenure group),0.793888,0.635484,0.526738,0.576023,0.834909,0.627081
Logistic Regression (without tenure group),0.803838,0.648485,0.572193,0.607955,0.835929,0.622817
Random Forest,0.784648,0.618729,0.494652,0.549777,0.813661,0.594200


## Model Comparison and Selection

Four approaches are compared on the same stratified train/test split: a most-frequent DummyClassifier, Logistic Regression with and without the engineered `tenure_group`, and Random Forest. Threshold-based metrics use the explicit rule `probability >= 0.5`; ROC-AUC and Average Precision evaluate ranking.

On this split, the refined Logistic Regression has the highest accuracy, precision, recall, F1, and ROC-AUC. The Logistic Regression with `tenure_group` has slightly higher Average Precision. These are small single-split differences, not evidence of reliable future performance. An operational model and decision threshold require validation on data representative of the intended prediction period and an agreed cost/capacity trade-off.

## Limitations and Next Steps

- Results use one stratified holdout split; cross-validation or a later-period test set is not included.
- The dataset is a customer snapshot. The prediction horizon and when each feature is available operationally are not defined; `TotalCharges` is cumulative and may not be suitable for every prediction time.
- No intervention costs, contact capacity, or retention experiment outcomes are available to choose an operational threshold or claim business impact.

A next evaluation should validate performance across folds or a later time period and select a threshold against explicit intervention capacity and error costs.

In [65]:
LogisticRegression(max_iter=2000)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

# 11. Project Summary

## Descriptive Findings

This dataset shows higher observed churn rates among month-to-month customers (42.7%), customers with 1–12 months of tenure (47.7%), electronic-check customers (45.3%), and customers without technical support (41.6%). Mean monthly charges are 74.44 for churned customers and 61.31 for retained customers. These are associations, not causal findings.

## Model Results

All models use the same stratified 80/20 split (random state 42) and an explicit probability threshold of 0.5 for threshold-based measures. Average Precision and ROC-AUC use probabilities.

| Model | Accuracy | Precision | Recall | F1 Score | ROC-AUC | Average Precision |
|---|---:|---:|---:|---:|---:|---:|
| Dummy (most frequent) | 73.42% | 0.00% | 0.00% | 0.00% | 50.00% | 26.58% |
| Logistic Regression (with `tenure_group`) | 79.39% | 63.55% | 52.67% | 57.60% | 83.49% | 62.71% |
| Logistic Regression (without `tenure_group`) | **80.38%** | **64.85%** | **57.22%** | **60.80%** | **83.59%** | 62.28% |
| Random Forest | 78.46% | 61.87% | 49.47% | 54.98% | 81.37% | 59.42% |

On this split, the refined Logistic Regression has the strongest threshold-based metrics and ROC-AUC, while the Logistic Regression with `tenure_group` has slightly higher Average Precision. The operating threshold should be chosen using intervention capacity and error costs, which are not available in this analysis.

# 12. Candidate Retention Experiments

The observed differences are associations, not evidence that the suggested actions cause churn to fall. Treat these as hypotheses for controlled testing rather than proven interventions.

## Hypothesis 1: Test Long-Term Contract Offers

Month-to-month customers have a higher observed churn rate. Test whether a contract offer changes retention, while accounting for offer eligibility and discount cost.

## Hypothesis 2: Test Early-Customer Support

Customers in their first year have a higher observed churn rate. Evaluate onboarding or early follow-up through a controlled experiment.

## Hypothesis 3: Investigate Payment Friction

Electronic-check customers have a higher observed churn rate than automatic-payment customers. Investigate billing experience before testing any payment-method intervention.

## Hypothesis 4: Test Proactive Technical Support

Customers without technical support have a higher observed churn rate. Test proactive support rather than assuming support adoption would reduce churn.

## Hypothesis 5: Investigate Higher-Charge Customers

Customers who churn have higher average monthly charges in this dataset. Assess satisfaction and service mix before testing targeted value offers.